# Sampling and evaluation

This notebook loads a checkpoint from [`05_train.ipynb`](05_train.ipynb), checks Algorithm 2, generates fixed-seed samples, measures held-out noise-prediction loss, and computes FID.

$$x_{t-1}=\frac{1}{\sqrt{\alpha_t}}\left(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\varepsilon_\theta(x_t,t)\right)+\sigma_tz.$$

The final evaluation uses the original 1,000-step DDPM sampler and reports FID-50k against the CIFAR-10 training split.


## 1. Setup


In [ ]:
import hashlib
import importlib
import math
import os
import random
import sys
import time
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

sys.modules["diffusion"] = importlib.import_module("02_diffusion")
sys.modules["model"] = importlib.import_module("04_unet")
from diffusion import build_schedule, p_sample, q_sample, sample_loop, simple_loss, to_display
from model import UNet

checkpoint_path = Path(
    os.getenv("DDPM_CHECKPOINT", "checkpoints/ddpm_cifar10_production_ema.pt")
)
if not checkpoint_path.exists():
    raise FileNotFoundError(f"Run 05_train.ipynb first: {checkpoint_path} does not exist")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for sampling")

payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
config_values = dict(payload["config"])
config_values["data_dir"] = Path(config_values["data_dir"])
config_values["checkpoint_dir"] = Path(config_values["checkpoint_dir"])
cfg = SimpleNamespace(**config_values)

device = torch.device("cuda")
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
torch.cuda.manual_seed_all(cfg.seed)

schedule = build_schedule(cfg.num_steps, cfg.beta_start, cfg.beta_end).to(device)
model = UNet(cfg).to(device).eval()
model.load_state_dict(payload["ema_model"])
global_step = int(payload["global_step"])
loss_history = list(payload.get("loss_history", []))

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
sampling_log = results_dir / "sampling_progress.log"
grid_size = 64
fid_samples = 50_000
sample_batch_size = int(os.getenv("DDPM_SAMPLE_BATCH", "128"))
snapshot_steps = tuple(round(value) for value in np.linspace(cfg.num_steps - 1, 0, 6))

print(f"Checkpoint: {checkpoint_path} (step {global_step})")
print(f"Device: {device} ({torch.cuda.get_device_name(0)})")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")
print(f"FID sample count: {fid_samples}; sample batch: {sample_batch_size}")


## 2. Sampler checks

One reverse step must preserve shape and stay finite. At `t=0` it must be deterministic.


In [ ]:
probe = torch.randn(2, cfg.in_channels, cfg.image_size, cfg.image_size, device=device)
mid_t = torch.full((2,), cfg.num_steps // 2, device=device, dtype=torch.long)
mid = p_sample(model, probe, mid_t, schedule)
g1 = torch.Generator(device=device).manual_seed(cfg.seed)
g2 = torch.Generator(device=device).manual_seed(cfg.seed)
mid1 = p_sample(model, probe, mid_t, schedule, g1)
mid2 = p_sample(model, probe, mid_t, schedule, g2)

last_t = torch.zeros(2, device=device, dtype=torch.long)
y1 = p_sample(model, probe, last_t, schedule)
y2 = p_sample(model, probe, last_t, schedule)

assert mid.shape == probe.shape and torch.isfinite(mid).all()
assert torch.equal(mid1, mid2) and torch.equal(y1, y2)
assert schedule.posterior_variance[0].item() == 0.0
print("One-step shape/finite and deterministic-generator checks passed.")


## 3. Training evidence

The full training checkpoint stores its loss history. The smaller EMA-only file stores only weights and step.


In [ ]:
if loss_history:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(loss_history, linewidth=0.8)
    ax.set(xlabel="optimizer step", ylabel="L_simple", title=f"Training loss, step {global_step}")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(results_dir / "training_loss_production.png", dpi=150)
    plt.show()
    print(f"L_simple: {loss_history[0]:.4f} -> {loss_history[-1]:.4f}")
else:
    print("This EMA-only checkpoint has no loss history.")


## 4. Fixed-seed samples

The 64-image grid uses a fixed seed for reproducibility. The trajectory shows the first generated image at six reverse steps.


In [ ]:
def generate_samples(count: int, keep_trajectory: bool = False):
    batches = []
    trajectory = {}
    started = time.perf_counter()
    for batch_index, start in enumerate(range(0, count, sample_batch_size)):
        size = min(sample_batch_size, count - start)
        generator = torch.Generator(device=device).manual_seed(cfg.seed + batch_index)
        with torch.autocast("cuda"):
            batch, snapshots = sample_loop(
                model,
                schedule,
                size,
                cfg.in_channels,
                cfg.image_size,
                device,
                snapshot_steps if keep_trajectory and batch_index == 0 else (),
                generator,
                show_progress=keep_trajectory and batch_index == 0,
            )
        batches.append(batch.cpu())
        if snapshots:
            trajectory = snapshots
        completed = start + size
        rate = completed / (time.perf_counter() - started)
        message = f"samples {completed}/{count}  {rate:.2f} images/s  ETA={(count - completed) / rate / 60:.0f} min"
        print(message)
        with sampling_log.open("a") as log_file:
            print(message, file=log_file)
    return torch.cat(batches), trajectory


fixed_seed_images, fixed_seed_trajectory = generate_samples(grid_size, keep_trajectory=True)
assert fixed_seed_images.shape == (grid_size, cfg.in_channels, cfg.image_size, cfg.image_size)
assert torch.isfinite(fixed_seed_images).all()

sample_grid = make_grid(to_display(fixed_seed_images), nrow=round(math.sqrt(grid_size)))
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(sample_grid.permute(1, 2, 0).numpy())
ax.set_title(f"EMA samples, step {global_step}")
ax.axis("off")
fig.tight_layout()
fig.savefig(results_dir / "samples_production.png", dpi=150)
plt.show()

steps = sorted(fixed_seed_trajectory, reverse=True)
fig, axes = plt.subplots(1, len(steps), figsize=(2 * len(steps), 2.3))
for ax, step in zip(axes, steps):
    ax.imshow(to_display(fixed_seed_trajectory[step][0]).permute(1, 2, 0).numpy())
    ax.set_title(f"t={step}")
    ax.axis("off")
fig.tight_layout()
fig.savefig(results_dir / "trajectory_production.png", dpi=150)
plt.show()


## 5. Held-out noise-prediction loss

The test split is never used for training or checkpoint selection. We report mean `L_simple` overall and in four timestep bands.


In [ ]:
test_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]
)
test_dataset = datasets.CIFAR10(cfg.data_dir, train=False, download=True, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False)
max_loss_batches = int(os.getenv("DDPM_LOSS_BATCHES", "0"))
losses = []
band_losses = [[] for _ in range(4)]

with torch.inference_mode():
    for batch_index, (images, _) in enumerate(test_loader):
        if max_loss_batches and batch_index >= max_loss_batches:
            break
        images = images.to(device)
        timesteps = torch.randint(0, cfg.num_steps, (images.size(0),), device=device)
        noise = torch.randn_like(images)
        prediction = model(q_sample(images, timesteps, schedule, noise), timesteps)
        per_image = (prediction - noise).square().flatten(1).mean(1)
        losses.extend(per_image.cpu().tolist())
        for band in range(4):
            selected = per_image[(timesteps * 4 // cfg.num_steps) == band]
            band_losses[band].extend(selected.cpu().tolist())

heldout_loss = float(np.mean(losses))
print(f"Held-out L_simple: {heldout_loss:.4f} over {len(losses)} images")
for band, values in enumerate(band_losses):
    low, high = band * cfg.num_steps // 4, (band + 1) * cfg.num_steps // 4 - 1
    print(f"t={low:>3}-{high:<3}: {np.mean(values):.4f}")


## 6. FID

FID-50k compares 50,000 generated images with all 50,000 CIFAR-10 training images, matching the paper's sample count and reference split. Feature extraction runs on CPU.


In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance

sampling_log.write_text("")
fake_samples, _ = generate_samples(fid_samples)
fake_uint8 = (to_display(fake_samples) * 255).round().to(torch.uint8)
real_dataset = datasets.CIFAR10(cfg.data_dir, train=True, download=True)
real_uint8 = torch.from_numpy(real_dataset.data).permute(0, 3, 1, 2)
fid_metric = FrechetInceptionDistance(feature=2048, normalize=False).cpu()
for start in range(0, fid_samples, 64):
    fid_metric.update(real_uint8[start : start + 64], real=True)
    fid_metric.update(fake_uint8[start : start + 64], real=False)
fid_score = float(fid_metric.compute())
print(f"FID-50k: {fid_score:.2f}")

with checkpoint_path.open("rb") as checkpoint_file:
    checkpoint_sha256 = hashlib.file_digest(checkpoint_file, "sha256").hexdigest()
metrics_text = (
    f"step={global_step}\nheldout_l_simple={heldout_loss:.6f}\n"
    f"fid-50k={fid_score:.4f}\ncheckpoint_bytes={checkpoint_path.stat().st_size}\n"
    f"checkpoint_sha256={checkpoint_sha256}\n"
)
(results_dir / "metrics_production.txt").write_text(metrics_text)


## 7. Recorded result

This summary is generated from the checkpoint and measurements above.


In [ ]:
print(f"Checkpoint step: {global_step}")
print(f"Held-out L_simple: {heldout_loss:.4f}")
print(f"FID-50k: {fid_score:.2f}")
print(f"EMA checkpoint: {checkpoint_path} ({checkpoint_path.stat().st_size / 1024**2:.1f} MiB)")
print(f"SHA-256: {checkpoint_sha256}")
print(f"Sample grid: {results_dir / 'samples_production.png'}")
print(f"Metrics: {results_dir / 'metrics_production.txt'}")


## 8. Interpretation and limits

- Final FID-50k: **PLEEASE UPD THIS**.
- Final sample quality: **PLEEASE UPD THIS**.
- The model uses the original 1,000-step DDPM sampler, a linear schedule, unconditional CIFAR-10, and a fixed 100,000-step training budget.
- The paper used the same sample count and reference split but trained for 800,000 steps and reported its best checkpoint.
- Class conditioning, classifier-free guidance, DDIM, and schedule ablations are outside this project's compute budget.
